In [ ]:
from experimental_pose_encoder_model_extension.advanced_pose_encoder import AdvancedPoseEncoder
from utils.evaluation.FGD.embedding_space_evaluator import EmbeddingSpaceEvaluator
import utils.utils as utils
from utils.animation.skeleton import Skeleton
from torch.utils.data import DataLoader
from dataset.dataset import *

with torch.no_grad():

    device = utils.get_device()

    # first we need to create the embedding space evaluator
    embeddingSpaceEvaluator = EmbeddingSpaceEvaluator(
        embed_net_path  = "utils/evaluation/FGD/models/fgd_model_30.pth",
        n_frames        = 30,
        device          = device,
        pose_dim        = 58 * 3
    )

    advanced_pose_encoder = AdvancedPoseEncoder.load_from_checkpoint("advanced_pose_encoder_ik_pca_64")

    val_loader = DataLoader(
        GPUDataset(
            # consolidated_file = "dataset/genea2023_dataset/val/main-agent/advanced_encoder/consolidated.npz",
            consolidated_file = "dataset/genea2023_dataset/val/main-agent/consolidated.npz",
            seq_length = 30,
            seed_length = 0,
            batch_size = 16384,
            epoch_length = 30,
            loading_encoded_data = True,
            device = device
        ),
        batch_size = 1,
        num_workers = 0,
        pin_memory = False
    )

    skeleton: Skeleton = val_loader.dataset.skeleton

    full_gesture_sequence, _, full_audio_features, main_agent_id_one_hot = [
        item.squeeze(0).float().to(device) for item in next(iter(val_loader))
    ]

    # decode the full gesture sequence
    # decoded_gesture_sequence = advanced_pose_encoder.decode(full_gesture_sequence)
    decoded_gesture_sequence = full_gesture_sequence

    # denormalize the decoded gesture sequence
    denormalized_gesture_seqquence = skeleton.denormalize_poses(decoded_gesture_sequence)

    world_pose_gesture_sequence = skeleton.calculate_world_positions(denormalized_gesture_seqquence)

    embeddingSpaceEvaluator.push_generated_samples(world_pose_gesture_sequence[8192:])

    embeddingSpaceEvaluator.push_real_samples(world_pose_gesture_sequence[:8192])

    fgd = embeddingSpaceEvaluator.get_fgd()

    print(f"FGD: {fgd}")


In [ ]:
import evaluation_fgd_calculation
from model import ContinuousMotionModel
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation

print(animation_visualisation.init_visualization(False))

val_loader = DataLoader(
        GPUDataset(
            consolidated_file = "dataset/genea2023_dataset/val/main-agent/advanced_encoder/consolidated.npz",
            # consolidated_file = "dataset/genea2023_dataset/val/main-agent/consolidated.npz",
            seq_length = 100,
            seed_length = 0,
            batch_size = 256,
            epoch_length = 1000,
            loading_encoded_data = True,
            device = device
        ),
        batch_size = 1,
        num_workers = 0,
        pin_memory = False
    )

model_path = utils.get_latest_model_path()
model_path = "trained_models/first_tests_2025-06-14_01-00-46/first_tests_2025-06-14_01-00-46_epoch_9.pth"
print(f"Loading model from {model_path}")
continous_motion_model = ContinuousMotionModel.load_model(model_path, device=device)

continous_motion_model.eval()

frechet_distance_feat_space, _ = evaluation_fgd_calculation.evaluate_frechet_gesture_distance(
    model = continous_motion_model,
    val_loader = val_loader,
    device = device,
    evaluation_length = 30,
    num_samples = 1024,
    calculate_raw_frechet_distance = False
)

print(frechet_distance_feat_space)